# 03 — CDC PLACES: County-Level Chronic Disease Prevalence

**Data source:** CDC PLACES (Population Level Analysis and Community Estimates)  
**What it contains:** County-level prevalence estimates for chronic diseases and health behaviors  
**Geographic grain:** County (FIPS) — joins directly onto CMS and SEER tables  
**DuckDB table written:** `cdc_places`  
**Why it matters:** Obesity, diabetes, and cardiovascular disease are primary comorbidities
that drive chronic venous disease and lymphedema. Counties with high chronic disease burden
= higher compression garment demand regardless of payer type.

### Measures we care about
| Measure | Why it matters |
|---|---|
| OBESITY | Obesity drives chronic venous insufficiency and lymphedema |
| DIABETES | Diabetic patients have high rates of peripheral edema and wound care needs |
| CHD | Coronary heart disease — cardiovascular comorbidity |
| STROKE | Post-stroke edema and lymphatic complications |
| CSMOKING | Current smoking — drives peripheral vascular disease |
| BPHIGH | High blood pressure — drives chronic venous hypertension (ICD I87.3) |


**DuckDB Table:** `cdc_places`

## Step 1 — Data Collection
Download county and census-tract chronic disease prevalence estimates from CDC PLACES.

In [3]:
import sys
import pandas as pd
import requests
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))

from config import DATA_RAW, DATA_PROCESSED, DB_PATH
from db import get_connection
from utils import ensure_dirs, log

ensure_dirs()
log("Notebook 03 — CDC PLACES — started")

[2026-04-19 19:11:27] INFO - Notebook 03 — CDC PLACES — started


In [5]:
# CDC PLACES uses the Socrata API on data.cdc.gov
# Dataset: PLACES County Data 2024 release
# No API key required — we add a small delay to be respectful
#
# We filter to only the 6 health measures we care about
# rather than pulling all 40+ measures in the dataset

PLACES_URL = "https://data.cdc.gov/resource/swc5-untb.json"

TARGET_MEASURES = [
    "OBESITY",    # Obesity among adults — drives venous disease and lymphedema
    "DIABETES",   # Diagnosed diabetes — peripheral edema and wound care comorbidity  
    "CHD",        # Coronary heart disease — cardiovascular comorbidity
    "STROKE",     # Stroke — post-stroke edema and lymphatic complications
    "CSMOKING",   # Current smoking — drives peripheral vascular disease
    "BPHIGH",     # High blood pressure — drives chronic venous hypertension (ICD I87.3)
]

# Test with a small request first — 5 rows, no filter
test_resp = requests.get(
    PLACES_URL,
    params={"$limit": 5, "$offset": 0},
    timeout=30
)

print("Status code:", test_resp.status_code)
print("Columns:", list(test_resp.json()[0].keys()))

Status code: 200
Columns: ['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', ':@computed_region_hjsp_umg2', ':@computed_region_skr5_azej']


In [6]:
def fetch_places_for_measure(measure_id: str) -> pd.DataFrame:
    """
    Fetch all county rows for a single CDC PLACES measure.
    
    Args:
        measure_id: CDC measure code e.g. 'OBESITY', 'DIABETES'
    Returns:
        DataFrame with one row per county for that measure
    """
    PAGE_SIZE = 50_000
    pages     = []
    offset    = 0

    while True:
        params = {
            "$where":  f"measureid='{measure_id}' AND datavaluetypeid='AgeAdjPrv'",
            "$select": "locationid,locationname,stateabbr,statedesc,measureid,short_question_text,data_value,totalpopulation",
            "$limit":  PAGE_SIZE,
            "$offset": offset,
        }

        resp = requests.get(PLACES_URL, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()

        if not batch:
            break

        pages.append(pd.DataFrame(batch))
        offset += PAGE_SIZE

        if len(batch) < PAGE_SIZE:
            break

    if not pages:
        log(f"WARNING: No data for measure {measure_id}", level="WARN")
        return pd.DataFrame()

    return pd.concat(pages, ignore_index=True)

# Test with one measure first
log("Testing fetch for OBESITY...")
test_df = fetch_places_for_measure("OBESITY")
print(f"Rows returned: {len(test_df):,}")
test_df.head(3)

[2026-04-19 19:12:07] INFO - Testing fetch for OBESITY...
Rows returned: 2,958


,locationid,locationname,stateabbr,statedesc,measureid,short_question_text,data_value,totalpopulation
0,13047,Catoosa,GA,Georgia,OBESITY,Obesity,33.9,68910
1,31135,Perkins,NE,Nebraska,OBESITY,Obesity,40.7,2795
2,32021,Mineral,NV,Nevada,OBESITY,Obesity,38.4,4528


In [7]:
# Fetch all 6 target measures and combine
all_frames = []

for i, measure in enumerate(TARGET_MEASURES):
    log(f"[{i+1}/{len(TARGET_MEASURES)}] Fetching measure: {measure}")
    df = fetch_places_for_measure(measure)
    if not df.empty:
        all_frames.append(df)
        log(f"  → {len(df):,} counties for {measure}")

places_raw = pd.concat(all_frames, ignore_index=True)

log(f"Total rows collected: {len(places_raw):,}")
log(f"Measures represented: {places_raw['measureid'].unique().tolist()}")

[2026-04-19 19:12:39] INFO - [1/6] Fetching measure: OBESITY
[2026-04-19 19:12:40] INFO -   → 2,958 counties for OBESITY
[2026-04-19 19:12:40] INFO - [2/6] Fetching measure: DIABETES
[2026-04-19 19:12:41] INFO -   → 2,958 counties for DIABETES
[2026-04-19 19:12:41] INFO - [3/6] Fetching measure: CHD
[2026-04-19 19:12:42] INFO -   → 2,958 counties for CHD
[2026-04-19 19:12:42] INFO - [4/6] Fetching measure: STROKE
[2026-04-19 19:12:43] INFO -   → 2,958 counties for STROKE
[2026-04-19 19:12:43] INFO - [5/6] Fetching measure: CSMOKING
[2026-04-19 19:12:43] INFO -   → 2,958 counties for CSMOKING
[2026-04-19 19:12:43] INFO - [6/6] Fetching measure: BPHIGH
[2026-04-19 19:12:44] INFO -   → 2,958 counties for BPHIGH
[2026-04-19 19:12:44] INFO - Total rows collected: 17,748
[2026-04-19 19:12:44] INFO - Measures represented: ['OBESITY', 'DIABETES', 'CHD', 'STROKE', 'CSMOKING', 'BPHIGH']


In [9]:
# ── Clean and pivot to wide format ─────────────────────────────────────────
# Current shape: one row per county per measure (long format)
# Target shape:  one row per county, one column per measure (wide format)

places = places_raw.copy()

# Cast numeric columns
places['data_value']      = pd.to_numeric(places['data_value'], errors='coerce')
places['totalpopulation'] = pd.to_numeric(places['totalpopulation'], errors='coerce')

# Normalize FIPS — must be 5-digit string with leading zeros
# locationid in CDC PLACES is missing leading zeros e.g. "1001" → "01001"
places['fips_county'] = places['locationid'].astype(str).str.zfill(5)

# Pivot wide — one row per county, one column per measure prevalence
places_wide = places.pivot_table(
    index=['fips_county', 'locationname', 'stateabbr', 'statedesc', 'totalpopulation'],
    columns='measureid',
    values='data_value',
    aggfunc='first'
).reset_index()

# Flatten column names
places_wide.columns.name = None
places_wide = places_wide.rename(columns={
    'locationname': 'county_name',
    'stateabbr':    'state_abbr',
    'statedesc':    'state_name',
    'totalpopulation': 'total_population',
    'OBESITY':   'pct_obese',           # % adults with obesity
    'DIABETES':  'pct_diabetes',        # % adults with diagnosed diabetes
    'CHD':       'pct_chd',             # % adults with coronary heart disease
    'STROKE':    'pct_stroke',          # % adults who have had a stroke
    'CSMOKING':  'pct_smoking',         # % adults who currently smoke
    'BPHIGH':    'pct_high_bp',         # % adults with high blood pressure
})

log(f"Wide format: {len(places_wide):,} counties, {len(places_wide.columns)} columns")
places_wide.head(3)

[2026-04-19 19:13:17] INFO - Wide format: 2,956 counties, 11 columns


,fips_county,county_name,state_abbr,state_name,total_population,pct_high_bp,pct_chd,pct_smoking,pct_diabetes,pct_obese,pct_stroke
0,01001,Autauga,AL,Alabama,60342,38.2,5.6,14.5,11.4,39.5,3.2
1,01003,Baldwin,AL,Alabama,253507,37.0,5.5,13.3,10.4,35.0,2.9
2,01005,Barbour,AL,Alabama,24585,45.4,7.4,20.8,16.5,47.1,4.7


In [11]:
# ── Add composite comorbidity score ────────────────────────────────────────
# A simple average of our 6 disease prevalence measures
# Higher score = higher chronic disease burden = higher compression garment demand
# This gives us a single number to rank counties by overall disease burden

places_wide['comorbidity_score'] = places_wide[[
    'pct_obese',
    'pct_diabetes',
    'pct_chd',
    'pct_stroke',
    'pct_smoking',
    'pct_high_bp',
]].mean(axis=1).round(2)

log(f"Comorbidity score range: {places_wide['comorbidity_score'].min():.1f} – {places_wide['comorbidity_score'].max():.1f}")
log(f"National average: {places_wide['comorbidity_score'].mean():.1f}")

# Save raw and processed
raw_out = DATA_RAW['cdc_places'] / 'cdc_places_county_raw.csv'
places_raw.to_csv(raw_out, index=False)
log(f"Raw saved → {raw_out}")

processed_out = DATA_PROCESSED['cdc_places'] / 'cdc_places_county_clean.csv'
places_wide.to_csv(processed_out, index=False)
log(f"Processed saved → {processed_out}")

# Write to DuckDB
con = get_connection()
con.execute("DROP TABLE IF EXISTS cdc_places")
con.execute("CREATE TABLE cdc_places AS SELECT * FROM places_wide")
count = con.execute("SELECT COUNT(*) FROM cdc_places").fetchone()[0]
log(f"cdc_places written to DuckDB: {count:,} rows")
con.close()

[2026-04-19 19:14:04] INFO - Comorbidity score range: 10.2 – 28.9
[2026-04-19 19:14:04] INFO - National average: 18.0
[2026-04-19 19:14:04] INFO - Raw saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\raw\cdc_places\cdc_places_county_raw.csv
[2026-04-19 19:14:04] INFO - Processed saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\processed\cdc_places\cdc_places_county_clean.csv
[2026-04-19 19:14:04] INFO - cdc_places written to DuckDB: 2,956 rows


In [12]:
# ── QA & summary stats ──────────────────────────────────────────────────────
con = get_connection()

print("=== Top 10 counties by comorbidity score ===")
print(con.execute("""
    SELECT 
        county_name,
        state_abbr,
        fips_county,
        pct_obese,
        pct_diabetes,
        pct_high_bp,
        comorbidity_score
    FROM cdc_places
    ORDER BY comorbidity_score DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Counties joining across all 3 datasets so far ===")
print(con.execute("""
    SELECT COUNT(DISTINCT c.fips_county) AS counties_in_all_3
    FROM cms_providers c
    INNER JOIN seer_cancer s ON c.fips_county = s.fips_county
    INNER JOIN cdc_places p  ON c.fips_county = p.fips_county
""").df().to_string(index=False))

con.close()


=== Top 10 counties by comorbidity score ===
  county_name state_abbr fips_county  pct_obese  pct_diabetes  pct_high_bp  comorbidity_score
Oglala Lakota         SD       46102       47.7          23.8         42.5              28.85
 East Carroll         LA       22035       51.7          21.3         52.5              28.43
         Todd         SD       46121       50.2          22.9         41.7              28.43
       Greene         AL       01063       53.4          21.1         52.4              27.50
      Bullock         AL       01011       51.0          21.3         52.4              27.43
    Humphreys         MS       28053       53.0          20.6         52.4              27.43
        Perry         AL       01105       54.0          20.5         51.9              27.28
       Holmes         MS       28051       51.5          21.0         53.1              27.22
        Sioux         ND       38085       49.7          20.1         41.8              26.90
      Madison  

## Notebook 03 complete ✓

### What we built
- Pulled 6 chronic disease prevalence measures for 2,956 counties from CDC PLACES
- Pivoted from long to wide format — one row per county, one column per measure
- Added composite comorbidity score (average of 6 measures) for county ranking
- Written to DuckDB table: `cdc_places`
- Saved to `data/processed/cdc_places/`

### Key findings
- National average comorbidity score: 18.0 (range 10.2–28.9)
- Highest burden counties: rural Deep South, Native American reservations
- Important: high score + low population = deprioritize despite high disease burden
- 1,875 counties now have data across CMS + SEER + CDC PLACES

### Comorbidity score interpretation
- Score < 15 = low chronic disease burden
- Score 15–22 = moderate — national average territory
- Score > 22 = high burden — prioritize if population